# Tarea 2 - Métodos determinísticos para la física

### Emil Winkler Partida - A01352501
### 11/03/2026

## Runge - Kutta de Cuarto orden 


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rk4(func, timesteps, initial_state, args = ()):
    y0 = np.asarray(initial_state) # Convierte el input en un array
    t0 , tf , h = timesteps

    t = np.linspace(t0, tf, int(abs(tf - t0)/h)+1)
    y = np.zeros((len(t), y0.size), dtype = y0.dtype)
    y[0] = y0


    for i in range(len(t) - 1):
        k1 = func(t[i], y[i], *args)
        k2 = func(t[i] + h/2, y[i] + h/2*(k1), *args)
        k3 = func(t[i] + h/2, y[i] + h/2*(k2), *args)
        k4 = func(t[i] + h, y[i] + h*k3, *args)

        y[i + 1] = y[i] + (h/6)*(k1 + 2*k2 + 2*k3 + k4)

    return t, y.T # Regresamos la matriz Transformada para el análisis físico

def añadir_val(msg, lim_inf, lim_sup):
    # Filtro para pedir número filtrando letras y rangos
    while True:
        try:
            valor = float(input(msg))
            if valor == 999:
                break
            # Filtro de rango máximo y mínimo
            if lim_inf <= valor <= lim_sup:
                print(f"Valor: {valor}")
                return valor
            else:
                print(f'El valor debe estar entre {lim_inf} y {lim_sup}. Vuelve a intentarlo')
        except ValueError: # Definimos un error en caso de que se introduzca algo erroneo
            print("Error: Ingresa un valor numérico válido, para salir de la interfaz poner 999")
    
def estado_inicial():
    print("CONFIGURACIÓN INICIAL DE PARTÍCULAS" + "\n")
    print("Las partículas deben estar separadas por lo menos por 3 metros")

    # Defimos listas vacias para los vectores de posicion y velocidad
    r = []
    v = []

    for i in range(3):
        print(f"Partícula {i +1}")

        # Aplicamos condicion que nos permite checar el filtro
        posicion_valida = False
        while not posicion_valida:
            x = añadir_val("Posición X (m) [-500, 500]: ", -500, 500)
            y = añadir_val("Posición Y (m) [-500, 500]: ", -500, 500)

            # Checamos que no haya una singularidad por Coulomb
            posicion_valida = True
            for j in range(i):
                # Checamos particulas ya guardadas
                x_prev = r[j*2]
                y_prev = r[j*2 + 1]
                # Analizamos distancia entre la particula anterior y la actual
                dist = np.sqrt((x - x_prev)**2 + (y - y_prev)**2)

                if dist < 3:
                    print(f"Alerta!! Muy cerca de la partícula {j +1}")
                    posicion_valida = False
                    break
        r.extend([x,y])

        # Limitamos velocidades iniciales
        vx = añadir_val("Velocidad X (m/s) [-50, 50]: ", -50, 50)
        vy = añadir_val("Velocidad Y (m/s) [-50, 50]: ", -50, 50)
        v.extend([vx,vy])
        print()

    print("Configuración validada. Iniciando simulación")
    return np.concatenate((r,v))


## Problema de las particulas 

Masa($m$): 11g = 0.011kg

Arrastre($\gamma$): 1 g/s = 0.001 kg/s

Potencial($k$): 0.001 N/m

Carga($q$): 0.001 C

Constante de Coulumb($K_e$): 8.987e9 $Nm^2/C^2$

Para lograr computar esto en nuestro método de RK4, vamos a tener que transformar nuestra función a algo que sea válido con los requisitos del método. 

Primero, vamos a definir un vector de valores iniciales en el cual, al ser un problema de condiciones iniciales de segundo orden nos obliga a tener dos condiciones iniciales por partícula. En este caso le asignaremos al vector de estados $y$ los valores de $x_i$ y $v_i$ de la forma siguiente:

$$ y = [x_1, y_1, x_2, y_1 x_3, y_3, v_{x1}, v_{y1}, v_{x2}, v_{y2}, v_{x3}, v_{y3}] $$

Asimismo somos concientes de como se defininen las derivadas en el movimiento

$$ \frac{dr}{dt} = v $$

$$ \frac{dv}{dt} = a $$

Finalmente definimos una función despejada para $a_i$ 

$$a_i = \frac{1}{m}\left(-\gamma v_i - kx_i + \sum_{j \ne i} \frac{K_e q_i q_j}{|r_{ij}|^2}\hat{r}_{ij}\right)$$

In [20]:
def particulas(t, y, m, k, gamma, q, Ke):

    # Tomamos los 6 primero elementos de los valores iniciales como posiciones 
    # y los siguienes 6 como velocidades. 
    r = y[0:6].reshape((3,2)) # Con reshape convertimos los convertimos en matrices de 3 filas
    v = y[6:12].reshape((3,2)) # Estas matrices tienen 3 particulas y 2 columnas con sus posiciones

    # Arreglos vacios para derivadas
    drdt = np.zeros((3,2))
    dvdt = np.zeros((3,2))

    # Calculamos las derivadas de particula por particula con un ciclo for
    for i in range (3):
        drdt[i] = v[i]

        # Fuerzas individuales que se aplican sobre la partícula
        F_armonica = -k * r[i]
        F_arrastre = -gamma * v[i]
        F_coulomb = np.zeros(2)  # Vector vacio para las fuerzas de Coulomb
        
        # Sumatoria de fuerzas producidas entre todas las particulas por Coulomb
        for j in range(3):
            if i != j: # No se pueden calcular la fuerza de Coulomb de la misma particula dos veces
                r_prima = r[i] - r[j] # Vector de distanca 
                dist = np.linalg.norm(r_prima)
                F_coulomb += Ke * (q**2) * r_prima / (dist**3)

        F_total = F_arrastre + F_armonica + F_coulomb # Sumamos las fuerzas
        dvdt[i] = F_total/m # Obtenemos la aceleración del despeje de la fuerza
    
    derivadas = np.concatenate((drdt.flatten(), dvdt.flatten()))
    
    return derivadas

### Puntos iniciales y graficación de partículas



In [ ]:
# Definimos los parámetros 
# (m, k, gamma, q, ke) en SI
parametros = (0.011, 0.001, 0.001, 0.001, 8.987e9)

# Le pedimos al usuario que ingrese el estado inicial
y0 = estado_inicial()

# Definimos el tiempo de la simulación 
tiempo_sim = (0, 500, 0.05)

# Sacamos los datos del método Runge Kutta
t, estados = rk4(func = particulas, timesteps = tiempo_sim, initial_state=y0, args = parametros)

# Gráficamos los resultados obtenidos por el método
x1, y1 = estados[0], estados[1]
x2, y2 = estados[2], estados[3]
x3, y3 = estados[4], estados[5]

plt.figure(figsize = (10,10))

plt.plot(x1, y1, color='blue', alpha=0.5, label='Trayectoria P1')
plt.plot(x2, y2, color='red', alpha=0.5, label='Trayectoria P2')
plt.plot(x3, y3, color='green', alpha=0.5, label='Trayectoria P3')

# Marcamos punto inicial
plt.scatter(x1[0], y1[0], color='blue', marker='*', s=250, edgecolor='black', zorder=5)
plt.scatter(x2[0], y2[0], color='red', marker='*', s=250, edgecolor='black', zorder=5)
plt.scatter(x3[0], y3[0], color='green', marker='*', s=250, edgecolor='black', zorder=5) 

# Marcamos con un círculo el punto final (Estado estacionario)
plt.scatter(x1[-1], y1[-1], color='blue', marker='o', s=150, edgecolor='black', zorder=5)
plt.scatter(x2[-1], y2[-1], color='red', marker='o', s=150, edgecolor='black', zorder=5)
plt.scatter(x3[-1], y3[-1], color='green', marker='o', s=150, edgecolor='black', zorder=5)

#Leyendas de la gráfica
plt.scatter([], [], color='black', marker='*', s=150, label='Punto de Inicio')
plt.scatter([], [], color='black', marker='o', s=150, label='Estado Final (Estacionario)')

plt.title('Trayectorias de 3 Partículas Atrapadas hasta el Estado Estacionario')
plt.xlabel('Posición X (m)')
plt.ylabel('Posición Y (m)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.axis('equal') # Mantiene la proporción 1:1 en los ejes
plt.show()

CONFIGURACIÓN INICIAL DE PARTÍCULAS

Las partículas deben estar separadas por lo menos por 3 metros
Partícula 1
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: Ingresa un valor numérico válido
Error: I

## Problema de circuitos

Para este problema nos encontramos con un desafío especial. En este caso las derivadas están acopladas, por los que le RK4 no las puede trabjar de manera directa. Por lo mismo, es nuestra obligación proporcionar un formáto que pueda ser analizable. 

Al igual que el problema anterior, en este caso contamos con un vector de estados $y$ de dos elementos fundamentales.

$$ y = [V_3, V_4] $$

Para aislar las ecuaciones, empezaremos a modificar algebraicamente las dos ecuaciones, comenzando con la segunda.

$$ \frac{dV_4}{dt} = R_4C_4(V_3 - V_4)$$

Ahora tomamos este resultado y lo sustituimos en la primera ecuación. 

$$ \frac{1}{R_2C_3} \frac{dV_3}{dt} [R_4C_4(V_3 - V_4)] + V_3 = V $$

De esta ecuación despejamos $\frac{dV_3}{dt}$ y con eso tendremos las dos ecuaciones para nuestro método RK4

$$ \frac{dV_3}{dt} = R_2C_3[V - V_3 - \frac{R_4}{R_2} (V_3 - V_4)$$

$$ \frac{dV_4}{dt} = R_2C_4(V_3 - V_4)

In [ ]:
def circuito(t, y, V, R2, R4, C3, C4):

    # Desampacamos vector de estados
    V3 = y[0]
    V4 = y[1]
    
    # Establecemos las ecuaciones diferenciales 
    dV3_dt = R2*C3*(V - V3 - (R4/R2) * (V3 - V4))
    dV4_dt = R4*C4*(V3 - V4)
    
    # Retornamos las ecuaciones como un arreglo de numpy
    return np.array([dV3_dt,dV4_dt])
